In [1]:
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


In [2]:
!mkdir -p ~/.kaggle
!cp '/gdrive/My Drive/Colab Notebooks/store_data/kaggle.json' ~/.kaggle/kaggle.json

In [3]:
!kaggle datasets download muhammadqasimshabbir/slideandseekclasificationlandslidedetectiondataset

Dataset URL: https://www.kaggle.com/datasets/muhammadqasimshabbir/slideandseekclasificationlandslidedetectiondataset
License(s): unknown
 99% 3.32G/3.36G [00:09<00:00, 489MB/s]
100% 3.36G/3.36G [00:09<00:00, 383MB/s]


In [4]:
!unzip -q   slideandseekclasificationlandslidedetectiondataset.zip

## Block 1: Import Libraries

In [5]:
# Import necessary libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


import torchvision.transforms as transforms
import torch
from torch.utils.data import TensorDataset, DataLoader,Dataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler

import albumentations as albu
import albumentations

from fastprogress import master_bar, progress_bar
import gc

from sklearn import metrics
import random
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

import timm

from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
from copy import deepcopy



In [6]:

def create_dir(path):
    if os.path.isdir(path)==False:
        os.makedirs(path)

def set_seed(seed = 0):
    '''Sets the seed of the entire notebook so results are the same every time we run.
    This is for REPRODUCIBILITY.'''
    np.random.seed(seed)
    random_state = np.random.RandomState(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    return random_state
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

def zeroonescale(img):
    ret=img.copy()
    ret=(ret-ret.min())/(ret.max()-ret.min())
    return ret
def zeroonescaleCH(img):
    ret=img.copy()
    for ch in range(img.shape[-1]):
        ret[...,ch]=zeroonescale(ret[...,ch])
    return ret

def robustCH(img, q05, q50, q95, channels=np.arange(12)):
    ret=img[...,channels].copy()
    for i, ch in enumerate(channels):
        ret[...,i]= (img[...,ch]-q50[ch])/(q95[ch]-q05[ch])
    return ret
def almost01CH(img, q05, q95, channels=np.arange(12)):
    ret=img[...,channels].copy()
    for i, ch in enumerate(channels):
        ret[...,i]= img[...,ch]/(q95[ch]-q05[ch])
    return ret
def ZeroOneCH(img, allmin, allmax, channels=np.arange(12)):
    ret=img[...,channels].copy()
    for i, ch in enumerate(channels):
        ret[...,i]= (img[...,ch]-allmin[ch])/(allmax[ch]-allmin[ch])
    return ret
def standardCH(img, allM, allS, channels=np.arange(12)):
    ret=img[...,channels].copy()
    for i, ch in enumerate(channels):
        ret[...,i]= (img[...,ch]-allM[ch])/(allS[ch])
    return ret

def find_opt_thresh(targets, preds):
    bestthresh=0.5
    bestscore=-np.inf
    for i in range(98):
        thresh=0.01+i/100
        score= f1_score(targets, (preds>thresh).astype(int))
        if score>bestscore:
            bestscore=score
            bestthresh=thresh
    return round(bestthresh,2), bestscore

In [7]:
name='Landslide_Class_S2S1_v31'

SIZE=256
BA=1
BS=32
N_folds=5
THRESH=0.5

In [24]:
# set the paths
TEST_PROBS_PATH='/gdrive/My Drive/Colab Notebooks/Landslide_class/testProbs/'
SUBMISSIONS_PATH='/gdrive/My Drive/Colab Notebooks/Landslide_class/submissions/'
OOF_PATH='/gdrive/My Drive/Colab Notebooks/Landslide_class/OOF/'
LOGS_PATH='/gdrive/My Drive/Colab Notebooks/Landslide_class/logs/'
weights_path='/gdrive/My Drive/Colab Notebooks/Landslide_class/weights/'

In [8]:
# Define paths for the dataset (remember to unzip the dataset first!)
train_csv_path = 'Train.csv'  # Path to the training labels CSV file
test_csv_path = 'Test.csv'    # Path to the test image IDs CSV file
train_data_path = 'train_data/train_data'  # Folder where .npy train files are located
test_data_path = 'test_data/test_data'    # Folder where .npy test files are located

# Load Train.csv and inspect the data
train_df = pd.read_csv(train_csv_path)
print("Train.csv:")
print(train_df.head())

Train.csv:
          ID  label
0  ID_HUD1ST      1
1  ID_KGE2HY      1
2  ID_VHV9BL      1
3  ID_ZT0VEJ      0
4  ID_5NFXVY      0


In [9]:
train_df.label.value_counts()

,count
label,
0,5892
1,1255


In [10]:
# Load all training data to ram
folder_path=train_data_path+'/'
X = np.array([np.load(folder_path+image_id+'.npy') for image_id in train_df['ID']])
y = train_df['label'].values

# Load all test data to ram
test_df = pd.read_csv(test_csv_path)
test_ids = test_df['ID'].values
X_test = np.array([np.load(test_data_path+'/'+image_id+'.npy') for image_id in test_ids])

In [11]:
# Q05=np.quantile(np.concatenate((X,X_test)),q=0.05,axis=(0,1,2))
# Q50=np.quantile(np.concatenate((X,X_test)),q=0.50,axis=(0,1,2))
# Q95=np.quantile(np.concatenate((X,X_test)),q=0.95,axis=(0,1,2))
# Q05,Q50,Q95

allM=np.mean(np.concatenate((X,X_test)), axis=(0,1,2))
allS=np.std(np.concatenate((X,X_test)), axis=(0,1,2))
allM, allS

(array([ 1.85185963e+03,  1.92838193e+03,  1.86725082e+03,  3.11747824e+03,
        -1.05585540e+01, -1.88282667e+01, -9.60078930e-01, -7.49858269e-01,
        -1.13484484e+01, -2.01162665e+01,  5.81196265e-01, -2.75804596e-01]),
 array([1341.69370135, 1273.92620502, 1278.76344252, 1466.91812707,
           8.34187043,   10.65859149,    4.33202752,    5.08455682,
           6.43914294,    9.9298614 ,    2.93451315,    4.50931721]))

In [12]:
X.shape, y.shape

((7147, 64, 64, 12), (7147,))

In [13]:
X.max(),X.min()

(np.float64(20944.0), np.float64(-65.66225893514942))

In [14]:
epochs=20
snapshots = 1
lr_0 = 0.00001
def cosine_anneal_schedule(t):
    cos_inner = np.pi * (t % (epochs // snapshots))
    cos_inner /= epochs // snapshots
    cos_out = np.cos(cos_inner) + 1
    return float(lr_0 / 2 * cos_out)
cosine_anneal_schedule_list=[cosine_anneal_schedule(x) for x in range(epochs)]

def adjust_optim(optimizer, n_iter):
    optimizer.param_groups[0]['lr'] = cosine_anneal_schedule_list[n_iter % epochs]


In [15]:
def feature_engineering(image):
    ndvi = (image[..., 3] - image[..., 0]) / (image[..., 3] + image[..., 0] + 1e-10)
    ndwi = (image[..., 3] - image[..., 1]) / (image[..., 3] + image[..., 1] + 1e-10)
    nirbleu = (image[..., 3] - image[..., 2]) / (image[..., 3] + image[..., 2] + 1e-10)

    bleugreen = (image[..., 2] - image[..., 1]) / (image[..., 2] + image[..., 1] + 1e-10)
    bleured = (image[..., 2] - image[..., 0]) / (image[..., 2] + image[..., 0] + 1e-10)
    greenred = (image[..., 1] - image[..., 0]) / (image[..., 1] + image[..., 0] + 1e-10)


    image = np.concatenate(
        [
            image,# * 2 - 1, # scale band data from -1 to 1
            np.expand_dims(ndvi+0.5, axis=-1),
            np.expand_dims(ndwi+0.5, axis=-1),
            np.expand_dims(nirbleu+0.5, axis=-1),

            np.expand_dims(bleugreen+0.5, axis=-1),
            np.expand_dims(bleured+0.5, axis=-1),
            np.expand_dims(greenred+0.5, axis=-1),

        ],
        axis=-1,
    )
    return image


In [16]:
channels=np.array([0,1,2,3, 6,7, 10,11])
print(channels)
NUM_CHANNELS=len(channels)

class Dataset_ram(torch.utils.data.Dataset):

    def __init__(self, iminds, transforms=None, preprocessing=None, mode='train'):
        self.iminds = iminds
        self.transforms = transforms
        self.preprocessing = preprocessing
        self.mode=mode

    def __len__(self):
        return len(self.iminds)

    def __getitem__(self, idx):
        if self.mode=='test':
            y_arr=np.zeros((512,512)).astype('uint8')
            x_arr=X_test[self.iminds[idx]]
        else:
            y_arr=y[self.iminds[idx]]
            x_arr=X[self.iminds[idx]]
        x_arr = standardCH(x_arr.astype('float32'), allM=allM, allS=allS)

        x_arr=x_arr[...,channels]

        y_arr = np.expand_dims(y_arr, -1)#.astype('float32')

        # Apply data augmentations, if provided
        if self.transforms:
            augmented = self.transforms(image=x_arr)
            x_arr = augmented['image']

        x_arr = np.transpose(x_arr, [2, 0, 1])

        return x_arr.astype('float32'), y_arr

[ 0  1  2  3  6  7 10 11]


In [17]:
training_transformations = albumentations.Compose(
    [
        albumentations.Resize(SIZE, SIZE, interpolation=1),
        albumentations.RandomRotate90(),
        albumentations.HorizontalFlip(),
        albumentations.VerticalFlip(),
    ]
)
validation_transformations = albumentations.Compose(
    [
        albumentations.Resize(SIZE, SIZE, interpolation=1),
    ]
)

In [18]:
ENCODER='vit_pwee_patch16_reg1_gap_256.sbb_in1k'

class combinemodels(nn.Module):
    def __init__(self):
        super(combinemodels, self).__init__()

        self.model1    = timm.create_model(ENCODER, in_chans=4,
                                                        pretrained=True, num_classes=256)
        self.model2    = timm.create_model(ENCODER, in_chans=4,
                                            pretrained=True, num_classes=256)

        self.feats = nn.Linear(in_features=256+256, out_features=256)
        self.bn = nn.BatchNorm1d(256)
        self.clf = nn.Linear(256*3, 1)


    def forward(self, x):
        # Concatenate along the time dimension
        rgbn = self.model1(x[:,:4,...])
        sar = self.model2(x[:,-4:,...])
        combined = torch.cat([rgbn, sar], 1)

        reducedfeats = self.feats(combined)
        reducedfeats = self.bn(reducedfeats)
        combined2 = torch.cat([combined, reducedfeats], 1)
        output = self.clf(combined2)

        return output
model=combinemodels()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/61.0M [00:00<?, ?B/s]

In [19]:
class SmoothF1Loss(nn.Module):
    def __init__(self, epsilon=1e-7):
        super(SmoothF1Loss, self).__init__()
        self.epsilon = epsilon

    def forward(self, logits, targets):
        """
        Computes a smooth F1 loss.

        Args:
            logits: Predicted raw outputs (before sigmoid), shape (batch_size,)
            targets: Ground truth binary labels, shape (batch_size,)

        Returns:
            Loss value to minimize (1 - soft F1 score)
        """
        # Apply sigmoid to convert logits to probabilities
        probs = torch.sigmoid(logits)

        # Ensure targets are float for arithmetic
        targets = targets.float()

        # True Positives, False Positives, False Negatives (soft)
        tp = (probs * targets).sum()
        fp = (probs * (1 - targets)).sum()
        fn = ((1 - probs) * targets).sum()

        # Smooth F1 calculation
        f1 = (2 * tp + self.epsilon) / (2 * tp + fp + fn + self.epsilon)

        # Convert to loss (1 - F1 to maximize F1)
        return 1 - f1

In [ ]:
basename=ENCODER+'_'+name

text_name = LOGS_PATH+name+'.txt'

OOF=np.zeros(len(y))

sgkf = KFold(n_splits=N_folds, random_state=42, shuffle=True)
for fold, (train_index, test_index) in enumerate(sgkf.split(y, y)):
    best_val_loss = 10000 #arbitrary large

    model=combinemodels()

    optimizer = optim.AdamW(model.parameters(), lr=lr_0)
    model.cuda()

    num_workers = 4

    train_dataset = Dataset_ram(iminds=train_index,
                                     transforms = training_transformations,
                                    )
    valid_dataset = Dataset_ram(iminds=test_index,
                                     transforms = validation_transformations,
                                     mode='eval'
                                    )

    train_loader = DataLoader(train_dataset, batch_size=BS, shuffle=True, num_workers=num_workers)#, drop_last=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BS, shuffle=False, num_workers=num_workers)

    criterion0 =  nn.BCEWithLogitsLoss()
    criterion = SmoothF1Loss()
    best_score=0.

    mb = master_bar(range(epochs))
    loss_log=[]
    seed_everything()
    for epoch in mb:

        avg_train_loss = 0.
        model.train()


        pb = progress_bar(train_loader)
        for ii, (data, target) in enumerate(pb):

            data, target = data.cuda(non_blocking=True), target.cuda(non_blocking=True)


            alpha = 1
            output = model(data)
            loss = criterion(output,  target.to(dtype=torch.float)) +criterion0(output,  target.to(dtype=torch.float))

            loss.backward()
            if (ii + 1) % BA == 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            if ii % 1000 == 0:
                loss_log.append(loss.item())
            avg_train_loss += loss.item() / len(train_loader)

        print('Epoch: {} - Loss: {:.6f}'.format(epoch + 1, avg_train_loss))


        model.eval()
        avg_val_loss = 0.

        val_preds, val_true =[], []
#         TP=0

        with torch.no_grad():
            pb = progress_bar(valid_loader)
            for i, (x_batch, y_batch) in enumerate(pb):
                x_batch=x_batch.cuda(non_blocking=True)
                y_batch=y_batch.cuda(non_blocking=True)

                preds = model(x_batch)

                loss = criterion(preds, y_batch.to(dtype=torch.float))

                preds=torch.sigmoid(preds)
                val_preds.append(preds.cpu())
                val_true.append(y_batch.cpu())


                avg_val_loss += loss.item() / len(valid_loader)


        print('Epoch: {} - Val Loss: {:.6f}'.format(epoch + 1, avg_val_loss))

        val_preds=np.vstack(val_preds)[:,0]
        val_true=np.vstack(val_true)[:,0]

        F1 = f1_score(val_true, (val_preds>THRESH).astype(int))
        print(F1)

        # Create log
        appendlogfile=('Epoch: '+str(epoch)+' - Loss: '+str(avg_train_loss)+
        '  Val Loss:' +str(avg_val_loss)+'  F1:' +str(F1) +'\n')
        with open(text_name, "a") as text_file:
            text_file.write(appendlogfile)

    #     scheduler.step()
        adjust_optim(optimizer, (epoch + 1))

        gc.collect()

    with open(text_name, "a") as text_file:
        text_file.write('\n')
    torch.save(model.state_dict(), weights_path+name+'_Fold'+str(fold)+'_last.pt')

    OOF[test_index]=val_preds
np.savez( OOF_PATH+name, OOF)


In [ ]:
f1_score(y, (OOF>THRESH).astype(int))

In [ ]:
bestthresh, bestscore=find_opt_thresh(y, OOF)
bestthresh, bestscore


In [ ]:
folder_path=test_data_path+'/'
# THRESH=bestthresh

test_dataset = Dataset_ram(iminds=np.arange(len(X_test)),
                                 transforms = validation_transformations,
                                 mode='test'
                                )

test_loader = DataLoader(test_dataset, batch_size=BS, shuffle=False, num_workers=num_workers)

all_test_preds= []
for fold in range(N_folds):

    model.load_state_dict(torch.load(weights_path+name+'_Fold'+str(fold)+'_last.pt'))
    test_preds= []

    with torch.no_grad():
        pb = progress_bar(test_loader)
        for i, (x_batch, y_batch) in enumerate(pb):

            preds0 = model(x_batch.cuda(non_blocking=True)).detach()
            preds1 = model(torch.flip(x_batch, dims=[2]).cuda(non_blocking=True)).detach()
            preds2 = model(torch.flip(x_batch, dims=[3]).cuda(non_blocking=True)).detach()
            preds3 = model(torch.flip(x_batch, dims=[2,3]).cuda(non_blocking=True)).detach()

            preds=(preds0 + preds1 + preds2 + preds3)/4

            preds=torch.sigmoid(preds)

            test_preds.append(preds.cpu())


    test_preds=np.vstack(test_preds)[:,0]
    all_test_preds.append(test_preds)
test_preds=np.vstack(all_test_preds)
test_preds=np.mean(test_preds,0)

np.savez( TEST_PROBS_PATH+name, test_preds)



# 0.5 threshold
test_preds01 = (test_preds>0.5).astype(int)

unique, counts = np.unique(test_preds01, return_counts=True)
prediction_counts = dict(zip(unique, counts))
print("Prediction counts:", prediction_counts)

# Prepare submission file
submission_df = pd.DataFrame({
    'ID': test_ids,
    'label': test_preds01.flatten() })
submission_df.to_csv(SUBMISSIONS_PATH+name+'THRESH'+str(0.5)+'.csv', index=False)


#bestThresh
test_preds01 = (test_preds>bestthresh).astype(int)
# Count the number of predictions for each class
unique, counts = np.unique(test_preds01, return_counts=True)
prediction_counts = dict(zip(unique, counts))
print("Prediction counts:", prediction_counts)

# Prepare submission file
submission_df = pd.DataFrame({
    'ID': test_ids,
    'label': test_preds01.flatten()  })
submission_df.to_csv(SUBMISSIONS_PATH+name+'THRESH'+str(bestthresh)+'.csv', index=False)


